# Validación en producción del modelo de predicción de retrasos

**TFM Cercanías Madrid · Máster en Data Science, Big Data & Business Analytics · UCM**

Este cuaderno construye el material gráfico de la validación realizada el 13 de septiembre
de 2026 sobre el servicio desplegado en `cercanias-madrid.es`.

## Qué se validó

Durante la tarde se consultó la aplicación cada veinte minutos sobre seis trayectos
de cinco líneas distintas, con instantes de consulta de hasta tres horas hacia delante, guardando cada predicción con su identificador de tren, su
horario oficial y el retraso previsto en segundos. Al día siguiente, cada uno de esos
trenes se buscó en las capturas del feed en tiempo real del operador para recuperar la
llegada realmente publicada.

## Qué NO es esto

No es una evaluación del modelo. Es una comprobación de comportamiento en producción
sobre **un solo día**, seis trayectos y cinco líneas. Sirve para detectar desviaciones
sistemáticas y para comprobar que el circuito completo funciona, no para estimar
métricas poblacionales.

## Sobre el dato de referencia

El operador no publica llegadas confirmadas a posteriori. Lo que se usa como valor real
es la **última estimación de llegada que el operador publicó antes de que el tren pasara
por la parada de destino**. Es el mejor dato disponible y la comparación debe leerse así.

In [ ]:
# Dependencias: pandas y matplotlib. Nada más.
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# --- Rutas -----------------------------------------------------------------
# El CSV lo genera tests/verificar_predicciones.py en el servidor. Se buscan
# varias ubicaciones para que el cuaderno funcione sin editar nada.
CANDIDATOS = [
    Path("validacion/verificacion.csv"),
    Path("../validacion/verificacion.csv"),
    Path("verificacion.csv"),
]
RUTA = next((p for p in CANDIDATOS if p.exists()), None)
if RUTA is None:
    raise FileNotFoundError(
        "No se encuentra verificacion.csv. Coloca el fichero junto al cuaderno "
        "o en la carpeta validacion/ del repositorio."
    )

FIGURAS = Path("figuras")
FIGURAS.mkdir(exist_ok=True)
print(f"Datos: {RUTA.resolve()}")

In [ ]:
import logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
# --- Estilo común -----------------------------------------------------------
# Tipografía alineada con la memoria (Verdana/Arial 10-11). Si no está
# disponible, matplotlib recurre a su fuente por defecto sin romper nada.
plt.rcParams.update({
    "font.family": ["Arial", "Liberation Sans", "Verdana", "DejaVu Sans"],
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linewidth": 0.6,
    "figure.dpi": 110,
    "savefig.dpi": 220,
    "savefig.bbox": "tight",
})

# Paleta sobria. Un color de acento para lo que se mide y un gris para el resto.
TINTA = "#101826"
ACENTO = "#e4572e"
APOYO = "#2a9d8f"
GRIS = "#8a94a6"
COLOR_LINEA = {
    "C1": "#4da8da", "C2": "#2aa36b", "C3": "#c9a227", "C4a": "#7b4b9c",
    "C4b": "#5b3070", "C5": "#e8833a", "C7": "#d1495b", "C8a": "#4da8da",
    "C8b": "#2c5f8a", "C10": "#8fbf3f",
}

def guardar(fig, nombre):
    """Guarda cada figura en PNG (memoria) y SVG (presentación, escala sin perder)."""
    for ext in ("png", "svg"):
        fig.savefig(FIGURAS / f"{nombre}.{ext}")
    print(f"  guardada figuras/{nombre}.png y .svg")

# --- Coma decimal ------------------------------------------------------------
# La memoria usa coma decimal. FMT se aplica a los ejes numéricos (no a los de
# categorías ni a los de fechas) y coma() a toda etiqueta con cifras.
def coma(valor, decimales=1):
    return f"{valor:.{decimales}f}".replace(".", ",").replace("-", "−")

FMT = mticker.FuncFormatter(lambda v, _: f"{v:g}".replace(".", ",").replace("-", "−"))


## 1. Carga y saneado

Dos filtros, y conviene entender por qué existen antes de mirar ningún número.

**`estado == "ok"`** descarta los trenes para los que no se pudo obtener referencia.

**`via == "parada"`** se queda solo con las predicciones cuya llegada real se leyó
directamente del campo `arrival` de la parada de destino. El resto se midió con una vía
de respaldo (el retraso del tren en una parada cualquiera) que resultó tener un error
medio de cuarenta minutos: no mide lo mismo y mezclarla contaminaría cualquier
estadístico. Se cuentan aparte como no medibles con fiabilidad.

In [ ]:
bruto = pd.read_csv(RUTA)
print(f"filas en el CSV: {len(bruto)}")
print(bruto["estado"].value_counts().to_string())

df = bruto[(bruto["estado"] == "ok") & (bruto["via"] == "parada")].copy()

# Conversiones a minutos: la memoria y la presentación hablan en minutos.
for col in ("retraso_predicho_s", "retraso_real_s", "error_retraso_s", "error_llegada_s"):
    df[col.replace("_s", "_min")] = df[col] / 60.0
df["error_abs_min"] = df["error_retraso_min"].abs()

for col in ("llegada_teorica_utc", "consultado_en_utc"):
    df[col] = pd.to_datetime(df[col])

descartadas = len(bruto[(bruto["estado"] == "ok") & (bruto["via"] != "parada")])
print(f"\npredicciones utilizables : {len(df)}")
print(f"descartadas por vía de respaldo: {descartadas}")
print(f"trenes distintos         : {df['nucleo'].nunique()}")
print(f"líneas                   : {sorted(df['line_id'].unique())}")

## 2. Cifras de cabecera

La tabla que va al cuerpo de la memoria. Se dan **mediana y media** juntas a propósito:
la distribución del error tiene cola larga, de modo que la media está tirada hacia arriba
por una minoría de casos y la mediana describe mejor lo que experimenta un viajero.

In [ ]:
def ficha(sub, etiqueta):
    return {
        "grupo": etiqueta,
        "n": len(sub),
        "mediana |error| (min)": round(sub["error_abs_min"].median(), 2),
        "media |error| (min)": round(sub["error_abs_min"].mean(), 2),
        "p90 |error| (min)": round(sub["error_abs_min"].quantile(0.9), 2),
        "sesgo medio (min)": round(sub["error_retraso_min"].mean(), 2),
        "dentro de ±3 min (%)": round(100 * (sub["error_abs_min"] <= 3).mean(), 1),
    }

filas = [ficha(df, "todas")]
for r, nombre in (("B", "aún no había salido"), ("A", "ya circulaba")):
    sub = df[df["regime"] == r]
    if len(sub):
        filas.append(ficha(sub, nombre))
for linea in sorted(df["line_id"].unique()):
    filas.append(ficha(df[df["line_id"] == linea], linea))

resumen = pd.DataFrame(filas).set_index("grupo")
resumen.to_csv(FIGURAS / "tabla_resumen.csv")
resumen

## 3. ¿Cuánto nos equivocamos?

Dos vistas del mismo dato. La **curva acumulada** es para la memoria: permite leer el
porcentaje de aciertos para cualquier umbral que se quiera discutir. Las **barras** son
para la presentación: dicen lo mismo sin exigir que nadie interprete un eje acumulado.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))

x = np.sort(df["error_abs_min"].values)
y = np.arange(1, len(x) + 1) / len(x) * 100
ax.plot(x, y, color=TINTA, linewidth=2)
ax.fill_between(x, 0, y, color=TINTA, alpha=0.07)

for umbral, color in ((3, ACENTO), (5, GRIS)):
    pct = (df["error_abs_min"] <= umbral).mean() * 100
    ax.axvline(umbral, color=color, linestyle="--", linewidth=1.2)
    ax.annotate(f"{pct:.0f} % dentro de ±{umbral} min",
                xy=(umbral, pct), xytext=(umbral + 0.6, pct - 12),
                color=color, fontweight="bold",
                arrowprops=dict(arrowstyle="-", color=color, linewidth=0.8))

ax.set_xlim(0, min(20, x.max()))
ax.set_ylim(0, 100)
ax.set_xlabel("Error absoluto de la predicción (minutos)")
ax.set_ylabel("Predicciones acumuladas (%)")
ax.set_title("Distribución acumulada del error")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.xaxis.set_major_formatter(FMT)
# Trenes y líneas se calculan: el 74 escrito a mano antes contaba también los 9 trenes
# que solo tenían medida de respaldo. Con la vía directa son 65.
fig.text(0.01, -0.04, f"n = {len(df)} predicciones · {df['nucleo'].nunique()} trenes · "
         f"{df['line_id'].nunique()} líneas · 13/09/2026",
         fontsize=8, color=GRIS)
guardar(fig, "01_acumulada_error")
plt.show()

In [ ]:
umbrales = [1, 2, 3, 5, 10]
pcts = [(df["error_abs_min"] <= u).mean() * 100 for u in umbrales]

fig, ax = plt.subplots(figsize=(7.2, 4.0))
barras = ax.bar([f"±{u} min" for u in umbrales], pcts,
                color=[ACENTO if u == 3 else GRIS for u in umbrales], width=0.6)
for b, p in zip(barras, pcts):
    ax.text(b.get_x() + b.get_width() / 2, p + 1.5, f"{p:.0f} %",
            ha="center", fontweight="bold", fontsize=11)

ax.set_ylim(0, 105)
ax.set_ylabel("Predicciones dentro del margen (%)")
ax.set_title("¿Con cuánto margen acierta la predicción?")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.grid(axis="x", visible=False)
guardar(fig, "02_margenes_acierto")
plt.show()

## 4. Predicho frente a real

El gráfico que más dice de todo el análisis. Cada punto es una predicción. La diagonal
sería la predicción perfecta; la banda gris es el margen de ±3 minutos.

Lo que hay que mirar no es la dispersión, sino **de qué lado de la diagonal cae la nube**.

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 6.2))

lim_min = min(df["retraso_predicho_min"].min(), df["retraso_real_min"].min()) - 2
lim_max = max(df["retraso_predicho_min"].max(), df["retraso_real_min"].max()) + 2

# Banda de ±3 minutos alrededor de la predicción perfecta.
rango = np.array([lim_min, lim_max])
ax.fill_between(rango, rango - 3, rango + 3, color=GRIS, alpha=0.13,
                label="margen de ±3 min")
ax.plot(rango, rango, color=TINTA, linewidth=1.2, linestyle="--",
        label="predicción perfecta")

for linea in sorted(df["line_id"].unique()):
    sub = df[df["line_id"] == linea]
    ax.scatter(sub["retraso_real_min"], sub["retraso_predicho_min"],
               s=26, alpha=0.65, edgecolor="none",
               color=COLOR_LINEA.get(linea, GRIS), label=linea)

ax.axhline(0, color=GRIS, linewidth=0.8)
ax.set_xlim(lim_min, lim_max)
ax.set_ylim(lim_min, lim_max)
ax.set_aspect("equal")
ax.xaxis.set_major_formatter(FMT)
ax.yaxis.set_major_formatter(FMT)
ax.set_xlabel("Retraso observado (minutos)")
ax.set_ylabel("Retraso predicho (minutos)")
ax.set_title("Predicho frente a observado")
ax.legend(frameon=False, fontsize=8, loc="upper left", ncol=2)

bajo = (df["retraso_predicho_min"] < df["retraso_real_min"]).mean() * 100
fig.text(0.01, -0.02,
         f"El {bajo:.0f} % de los puntos cae por debajo de la diagonal: "
         f"el modelo predice menos retraso del observado.",
         fontsize=8, color=GRIS)
guardar(fig, "03_predicho_vs_real")
plt.show()

## 5. Dos distribuciones que no se parecen

Aquí está la explicación visual del sesgo. El modelo produce una banda estrecha de
valores; la realidad tiene una cola larga por la derecha que el modelo no alcanza.

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 4.2))

bins = np.arange(-10, 32, 1.5)
ax.hist(df["retraso_real_min"], bins=bins, color=ACENTO, alpha=0.55,
        label="observado", edgecolor="white", linewidth=0.5)
ax.hist(df["retraso_predicho_min"], bins=bins, color=TINTA, alpha=0.55,
        label="predicho", edgecolor="white", linewidth=0.5)

# La media predicha se rotula a la izquierda de su línea y la observada a la
# derecha: están a menos de 4 minutos y a la misma altura se solapaban.
for serie, color, nombre, lado in ((df["retraso_real_min"], ACENTO, "observado", 1),
                                   (df["retraso_predicho_min"], TINTA, "predicho", -1)):
    m = serie.mean()
    ax.axvline(m, color=color, linestyle="--", linewidth=1.4)
    ax.annotate(f"media {nombre}\n{coma(m)} min", xy=(m, ax.get_ylim()[1] * 0.86),
                xytext=(m + 0.6 * lado, ax.get_ylim()[1] * 0.86),
                ha="left" if lado > 0 else "right",
                color=color, fontsize=8, fontweight="bold")
ax.xaxis.set_major_formatter(FMT)

ax.set_xlabel("Retraso (minutos)")
ax.set_ylabel("Número de predicciones")
ax.set_title("Distribución del retraso: predicho frente a observado")
ax.legend(frameon=False)
guardar(fig, "04_distribuciones")
plt.show()

## 6. El sesgo por línea

En media, el modelo predice menos retraso del observado en las cinco líneas. Pero la
media está arrastrada por la cola de trenes muy retrasados. La **mediana**, que describe
al tren típico, cuenta otra cosa: el sesgo va de casi cero en la C4a a más de siete
minutos en la C7, y crece con la longitud del recorrido.

La causa identificada del sesgo global es que casi todas las predicciones de este día
se hicieron fuera del dominio de entrenamiento (consultas con más de 30 minutos de
antelación sobre la salida del tren). Se documenta en el informe de validación.


In [ ]:
por_linea = (df.groupby("line_id")
               .agg(sesgo_medio=("error_retraso_min", "mean"),
                    sesgo_mediano=("error_retraso_min", "median"),
                    n=("error_retraso_min", "size"))
               .sort_values("sesgo_medio"))

fig, ax = plt.subplots(figsize=(7.2, 3.8))
y = np.arange(len(por_linea))
ax.barh(y, por_linea["sesgo_medio"], height=0.55,
        color=[COLOR_LINEA.get(l, GRIS) for l in por_linea.index], alpha=0.85,
        label="media")
ax.scatter(por_linea["sesgo_mediano"], y, color=TINTA, zorder=3, s=40,
           marker="D", label="mediana")

ax.axvline(0, color=TINTA, linewidth=1.2)
ax.set_yticks(y)
ax.set_yticklabels([f"{l}  (n={n})" for l, n in zip(por_linea.index, por_linea["n"])])
ax.set_xlabel("Sesgo: predicho − observado (minutos)")
ax.set_title("El sesgo del tren típico cambia mucho entre líneas")
# Leyenda fuera del área de datos: dentro tapaba la barra de la C7.
ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper center",
          bbox_to_anchor=(0.5, -0.18))
ax.xaxis.set_major_formatter(FMT)
ax.grid(axis="y", visible=False)
guardar(fig, "05_sesgo_por_linea")
plt.show()

## 7. Error según el horizonte de predicción

Cuánto falta para la llegada cuando se hace la predicción. Con **medianas**, el error
apenas cambia entre una y tres horas de horizonte; lo que crece es la dispersión. Una
lectura con medias sugería que el error bajaba al alargar el horizonte: era un artefacto
de los valores extremos.

El eje vertical se recorta en el percentil 97 del error para que se lean las cajas.


In [ ]:
cortes = [0, 30, 60, 90, 120, 180, 10**6]
etiquetas = ["<30", "30-60", "60-90", "90-120", "120-180", ">180"]
df["tramo_horizonte"] = pd.cut(df["horizonte_min"], bins=cortes, labels=etiquetas,
                               right=False)

grupos = [df[df["tramo_horizonte"] == e]["error_abs_min"].values for e in etiquetas]
presentes = [(e, g) for e, g in zip(etiquetas, grupos) if len(g) > 0]

fig, ax = plt.subplots(figsize=(7.2, 4.2))
bp = ax.boxplot([g for _, g in presentes], labels=[e for e, _ in presentes],
                patch_artist=True, showfliers=False, widths=0.55,
                medianprops=dict(color=ACENTO, linewidth=2))
for caja in bp["boxes"]:
    caja.set(facecolor=TINTA, alpha=0.18, edgecolor=TINTA)

for i, (_, g) in enumerate(presentes, start=1):
    ax.scatter(np.random.normal(i, 0.06, len(g)), g, s=8, color=TINTA, alpha=0.25)
    ax.text(i, -0.9, f"n={len(g)}", ha="center", fontsize=8, color=GRIS)

ax.axhline(3, color=GRIS, linestyle="--", linewidth=1)
ax.set_ylim(-1.6, min(20, df["error_abs_min"].quantile(0.97)))
ax.set_xlabel("Horizonte de predicción (minutos hasta la llegada teórica)")
ax.set_ylabel("Error absoluto (minutos)")
ax.set_title("Error según cuánto falta para la llegada")
ax.grid(axis="x", visible=False)
ax.yaxis.set_major_formatter(FMT)
fig.text(0.01, -0.03, "Eje vertical recortado en el percentil 97 del error.", fontsize=8, color=GRIS)
guardar(fig, "06_error_por_horizonte")
plt.show()

## 8. Seguimiento de un tren concreto

El gráfico para la defensa. Un único tren, predicho varias veces a lo largo de la tarde
según se acercaba su hora de llegada, contrastado con la llegada que el operador acabó
publicando.

Se elige automáticamente el tren con más predicciones y mayor recorrido de horizonte.

In [ ]:
conteo = (df.groupby("nucleo")
            .agg(n=("error_abs_min", "size"),
                 recorrido=("horizonte_min", lambda s: s.max() - s.min()))
            .sort_values(["n", "recorrido"], ascending=False))
elegido = conteo.index[0]
tren = df[df["nucleo"] == elegido].sort_values("consultado_en_utc")
info = tren.iloc[0]

fig, ax = plt.subplots(figsize=(7.6, 4.4))
horas = tren["consultado_en_utc"].dt.tz_convert("Europe/Madrid")

ax.plot(horas, tren["retraso_predicho_min"], marker="o", color=TINTA,
        linewidth=2, markersize=6, label="retraso predicho")
ax.axhline(info["retraso_real_min"], color=ACENTO, linewidth=2, linestyle="--",
           label=f"retraso observado ({coma(info['retraso_real_min'])} min)")
ax.fill_between(horas, info["retraso_real_min"] - 3, info["retraso_real_min"] + 3,
                color=ACENTO, alpha=0.10, label="margen de ±3 min")

for _, fila in tren.iterrows():
    if fila["regime"] == "A":
        ax.scatter(fila["consultado_en_utc"].tz_convert("Europe/Madrid"),
                   fila["retraso_predicho_min"], s=140, facecolor="none",
                   edgecolor=APOYO, linewidth=1.8, zorder=4)
# La entrada de leyenda solo se pinta si alguna predicción del tren fue en marcha.
if (tren["regime"] == "A").any():
    ax.scatter([], [], s=140, facecolor="none", edgecolor=APOYO, linewidth=1.8,
               label="el tren ya circulaba")

# El eje es el INSTANTE CONSULTADO, que puede ser futuro: la captura del 13/09
# pedía también predicciones a +15, +30, +60, +120 y +180 minutos.
ax.set_xlabel("Instante consultado (hora de Madrid)")
import matplotlib.dates as mdates
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz="Europe/Madrid"))
ax.yaxis.set_major_formatter(FMT)
ax.set_ylabel("Retraso (minutos)")
ax.set_title(f"Evolución de la predicción · {info['line_id']} "
             f"{info['origen_nombre']} → {info['destino_nombre']}")
ax.legend(frameon=False, fontsize=8)
fig.autofmt_xdate()
fig.text(0.01, -0.03,
         f"Llegada teórica a las {info['llegada_teorica_utc'].tz_convert('Europe/Madrid'):%H:%M} (hora de Madrid) · "
         f"{len(tren)} predicciones a lo largo de la tarde",
         fontsize=8, color=GRIS)
guardar(fig, "07_seguimiento_de_un_tren")
plt.show()

## 9. Cobertura: de dónde salen los datos

Antes de cualquier métrica hay que contar qué se pudo medir y qué no. Un tren que no
aparece en el feed no es un error del modelo, y mezclar ambas cosas invalidaría la media.

In [ ]:
n_total = len(bruto)
n_ok = len(bruto[bruto["estado"] == "ok"])
n_directa = len(df)
otros = bruto[bruto["estado"] != "ok"]["estado"].value_counts()

etapas = ["Predicciones\ncapturadas", "Con referencia\nen el feed",
          "Medidas por\nvía directa"]
valores = [n_total, n_ok, n_directa]

fig, ax = plt.subplots(figsize=(7.0, 4.0))
barras = ax.bar(etapas, valores, color=[GRIS, GRIS, ACENTO], width=0.55)
for b, v in zip(barras, valores):
    ax.text(b.get_x() + b.get_width() / 2, v + n_total * 0.02, f"{v}",
            ha="center", fontweight="bold")
    ax.text(b.get_x() + b.get_width() / 2, v / 2, f"{100*v/n_total:.0f} %",
            ha="center", color="white", fontweight="bold")

ax.set_ylim(0, n_total * 1.15)
ax.set_ylabel("Número de predicciones")
ax.set_title("Cobertura del contraste")
ax.grid(axis="x", visible=False)
guardar(fig, "08_cobertura")
plt.show()

print("Descartes:")
for estado, n in otros.items():
    print(f"  {estado}: {n}")
print(f"  medidas por vía de respaldo (excluidas): {n_ok - n_directa}")

## 10. ¿Ayuda conocer el retraso del propio tren?

Cuando el tren ya circula, el modelo dispone de su retraso acumulado, que es la variable
con mayor peso del conjunto. La comparación entre ambos regímenes es, con esta muestra,
**inconcluyente**: los tamaños son demasiado pequeños para afirmar nada.

Se incluye porque la ausencia de efecto es en sí misma un resultado que conviene
investigar, y porque enseñar una comparación sin significación es más honesto que
omitirla.

In [ ]:
regimenes = [("B", "Aún no había salido"), ("A", "Ya circulaba")]
datos = [(nombre, df[df["regime"] == r]["error_abs_min"].values)
         for r, nombre in regimenes]
datos = [(n, v) for n, v in datos if len(v) > 0]

fig, ax = plt.subplots(figsize=(6.4, 4.0))
for i, (nombre, valores) in enumerate(datos):
    ax.scatter(np.random.normal(i, 0.07, len(valores)), valores,
               s=22, alpha=0.45, color=TINTA, edgecolor="none")
    ax.hlines(np.median(valores), i - 0.28, i + 0.28, color=ACENTO, linewidth=2.5)
# El tamaño y la mediana van en la etiqueta del eje: dentro del gráfico
# quedaban debajo de los puntos y no se leían.
ax.set_xticks(range(len(datos)))
ax.set_xticklabels([f"{n}\nn = {len(v)} · mediana {coma(np.median(v))} min"
                    for n, v in datos])
ax.set_ylim(0, min(20, df["error_abs_min"].quantile(0.97)))
ax.yaxis.set_major_formatter(FMT)
fig.text(0.01, -0.06, "Eje vertical recortado en el percentil 97 del error.", fontsize=8, color=GRIS)
ax.set_ylabel("Error absoluto (minutos)")
ax.set_title("Error según el régimen del tren")
ax.grid(axis="x", visible=False)
guardar(fig, "09_regimen")
plt.show()

## 11. Qué llevarse de aquí

**El resultado positivo.** Algo más de la mitad de las predicciones acierta dentro de un
margen de tres minutos, con un error mediano en torno a los tres minutos.

**El resultado incómodo, que es el más valioso.** El modelo comprime el rango: no predice
los retrasos grandes, y una regla constante de cuatro minutos obtiene un error medio
menor. La causa identificada es que el 95 % de las predicciones de este día quedó fuera
del dominio de entrenamiento, porque la captura consultaba trenes con horas de antelación
y el modelo solo se entrenó con consultas de hasta 30 minutos antes de la salida. La
aplicación limita ahora las predicciones a ese dominio. Queda abierto un segundo
desajuste: la variable de mayor peso se calcula en servicio sobre el retraso que publica
el operador y en entrenamiento sobre el retraso reconstruido.

**Lo que esta validación demuestra sobre el método.** Un modelo evaluado solo contra su
conjunto de prueba no puede detectar una diferencia entre cómo se construyen sus datos
al entrenar y cómo se construyen en servicio. Hizo falta contrastarlo en producción para
que se hiciera visible.

---

### Uso de las figuras

- **Memoria y anexos**: los PNG a 220 ppp.
- **Presentación**: los SVG, que escalan sin pérdida.
- La figura 04 repite el contenido de `f1_compresion.png` del cuaderno del anexo; se
  conserva por compatibilidad, pero en el anexo se usa la otra.
